# OpenAI API 구조 실습

A-PI 사전조사 — OpenAI 를 7개 항목으로 실측하고 `evidence/openai/*.json` 을 남기는 노트북(증거 생성기).
어댑터 `apicompare.providers.openai_adapter.OpenAIAdapter` 를 얇게 호출하고, 각 섹션 끝에서 `save_evidence()` 로 결과를 저장한다.

**규칙**
- 키는 `.env` 에만. 노트북 안에서 `print(client)`, `os.environ` 전체 출력, 키 출력 **금지**.
- 키가 없으면(`NO_KEY`) 모든 섹션이 `[BLOCKED: no key]` 를 출력하고 `source="blocked"` 로 evidence 를 남긴다(에러 없이 끝까지 실행된다).
- 유료 셀은 `💸` 로 표시했다. 전부 `max_output_tokens <= 64`, 모델은 `gpt-5.4-mini`(총 3회 호출, 약 $0.0003 미만).
- 출력(outputs)은 커밋하지 않는다(`.gitattributes` → nbstripout). "관찰 메모" 는 실행 후 손으로 채운다.

| § | 항목 | 어댑터 메서드 | 비용 |
|---|---|---|---|
| 1 | 모델 목록 조회 | `list_models()` | 무료 |
| 2 | 모델 호출 | `generate(api="responses" / "chat_completions")` | 💸 2회 |
| 3 | Token Usage | `generate().usage`, `count_tokens()` | 무료(§2 결과 재사용) |
| 4 | Usage 조회 | `usage_report()`, `probe_usage_api_with_standard_key()` | 무료 |
| 5 | 비용 확인 | `cost()`, `cost_report()` | 무료 |
| 6 | 상태 및 오류 확인 | `health()`, `status_page()`, 의도적 오류 4종 | 무료(오류는 과금 전 거부) |
| 7 | 호출 예제 | SDK 직접 호출 + curl | 💸 1회 |

## 0. 준비

- 실행 환경: 저장소 루트에서 `uv sync --group dev` → `uv run jupyter lab` (Python 3.12, `openai` 3.35.1).
- `.env` 의 `OPENAI_API_KEY`(표준/프로젝트 키)는 `apicompare.config.load_env()` 로 읽는다. `OPENAI_ADMIN_KEY` 는 §4·§5 의 Admin API 전용(없어도 된다).
- 어댑터 기본값: `timeout_s=30`, `max_retries=0`(SDK 기본은 600초·2회). 세 Provider 의 응답시간 비교를 왜곡하지 않기 위해 재시도를 끈다.
- 출처: https://github.com/openai/openai-python · https://developers.openai.com/api/docs/quickstart

In [ ]:
import sys, json
assert sys.version_info >= (3, 12), "Python 3.12+ 가 필요하다. 저장소 루트에서 `uv run jupyter lab` 으로 실행했는지 확인."

import openai
import pandas as pd
from apicompare.config import load_env, keys
load_env()  # .env → 환경변수 (값은 출력하지 않는다)

from apicompare.providers import get_adapter
from apicompare.evidence import save_evidence, summary
from apicompare.models import NotSupported, to_dict
from apicompare import usagelog, pricing

print("python", sys.version.split()[0], "| openai SDK", openai.__version__)
print("키 존재 여부(bool):", keys())

adapter = get_adapter("openai")            # timeout_s=30, max_retries=0
MODEL = adapter.default_model              # gpt-5.4-mini
MODE = adapter.mode                        # LIVE | NO_KEY
BLOCKED = MODE == "NO_KEY"
PROVIDER = "openai"
print(f"\n=== [{MODE}] provider={PROVIDER} model={MODEL} timeout_s={adapter.timeout_s} max_retries={adapter.max_retries} ===")
if BLOCKED:
    print("OPENAI_API_KEY 가 없어 모든 섹션이 [BLOCKED: no key] 로 기록된다. .env 를 채우고 커널을 재시작하면 LIVE 로 바뀐다.")

def show(obj, limit=3000):
    print(json.dumps(to_dict(obj), ensure_ascii=False, indent=2, default=str)[:limit])

def blocked_banner(item):
    print(f"[BLOCKED: no key] {item} — OPENAI_API_KEY 없음. 건너뜀.")

def save_blocked(item):
    path = save_evidence(PROVIDER, item, {"error": to_dict(adapter.no_key_error())}, "blocked", note="no key", model=MODEL)
    print("[BLOCKED: no key] evidence ->", path)

In [ ]:
# 설정값 확인 (키 값·client 객체는 출력하지 않는다)
print({"name": adapter.name, "key_env": adapter.key_env, "admin_key_env": adapter.admin_key_env,
       "default_model": adapter.default_model, "mode": MODE, "has_admin_key": bool(adapter.admin_key)})

## 1. 모델 목록 조회

- **공식 방법**: `GET https://api.openai.com/v1/models` (SDK `client.models.list()`), 개별 모델은 `GET /v1/models/{model}`. 표준 API 키.
- **응답 필드**: `object="list"`, `data[].{id, object="model", created(Unix초), owned_by, shutdown_date(종료 예정일 또는 null)}` — SDK 3.35.1 `types/model.py` 의 `Model` 에 `shutdown_date` 필드가 있다.
- **주의점**: 컨텍스트 윈도우·최대 출력·단가·capability 는 **API 로 제공되지 않는다**(Gemini `models.list` 와 다름) → 모델 스펙은 문서(`developers.openai.com/api/docs/models/{id}`)에서 로컬 테이블로 관리. TTS/임베딩/이미지/음성 모델이 섞여 있어 텍스트 모델만 필터링해야 한다(`adapter.is_text_model`). 종료 일정: gpt-5-mini/gpt-5-nano 2026-12-11, gpt-4o-mini/gpt-4.1 2027-01-14, gpt-5.4-nano 2027-04-01.
- **출처**: https://developers.openai.com/api/reference/resources/models/methods/list · https://developers.openai.com/api/docs/models · https://developers.openai.com/api/docs/deprecations

In [ ]:
if BLOCKED:
    blocked_banner("① 모델 목록 조회"); models, text_models = [], []
else:
    models = adapter.list_models()                       # GET /v1/models (무료)
    if not models and adapter.last_error:
        print("list_models 실패:", to_dict(adapter.last_error))
    text_models = [m for m in models if adapter.is_text_model(m.id)]
    print(f"전체 {len(models)}개 / 텍스트 생성 모델 {len(text_models)}개 (tts·embedding·dall-e·whisper·image·realtime·transcribe 제외)")
    df_models = pd.DataFrame([{"id": m.id, "created_at": m.created_at, "shutdown_date": m.shutdown_date, "owned_by": m.capabilities.get("owned_by")} for m in text_models])
    display(df_models.head(40))

In [ ]:
# 원본 객체/필드 확인: Model 객체에 어떤 키가 있는지 (컨텍스트·단가 필드가 없음을 확인)
if BLOCKED:
    blocked_banner("① raw 필드")
else:
    print("raw keys:", sorted(models[0].raw.keys()) if models else None)
    print("default_model 포함 여부:", any(m.id == MODEL for m in models))
    with_shutdown = [(m.id, m.shutdown_date) for m in models if m.shutdown_date]
    print(f"shutdown_date 가 있는 모델 {len(with_shutdown)}개 (앞 15개):")
    for mid, d in sorted(with_shutdown, key=lambda x: x[1])[:15]:
        print(f"  {d}  {mid}")

**관찰 메모: (실행 후 채우기)**
- 전체 모델 수 / 텍스트 모델 수:
- `gpt-5.4-mini` 가 목록에 있는가, `shutdown_date` 는:
- 컨텍스트 윈도우·단가 필드가 없음을 확인했는가:

In [ ]:
if BLOCKED:
    save_blocked("01_models")
else:
    payload = {"count": len(models), "text_model_count": len(text_models),
               "raw_keys": sorted(models[0].raw.keys()) if models else [],
               "models": [to_dict(m) for m in text_models[:80]]}
    print("evidence ->", save_evidence(PROVIDER, "01_models", payload, "live" if models else "blocked",
                                       note="" if models else (adapter.last_error.category if adapter.last_error else "empty"), model=MODEL))

## 2. 모델 호출

- **공식 방법(두 엔드포인트 공존)**
  - `POST /v1/responses` — **신규 프로젝트 권장**. `input`(문자열 또는 메시지 배열), `instructions`(시스템 메시지), `max_output_tokens`(최소 16), `store`(기본 true, 30일 보관 → 비교 실험은 `store=False`), `reasoning={"effort": ...}`, Structured Outputs 는 `text={"format": {"type": "json_schema", ...}}`. 결과 `response.output_text`(SDK 편의 속성), `status`, `incomplete_details.reason`, `usage`.
  - `POST /v1/chat/completions` — 계속 지원(종료 미공지). `messages=[{role, content}]`, `max_completion_tokens`(`max_tokens` 는 deprecated), Structured Outputs 는 `response_format`. 결과 `choices[0].message.content`, `choices[0].finish_reason`.
- **주의점**: GPT-5.4 이후 Chat Completions 는 `tools` + `reasoning_effort≠none` 조합을 지원하지 않아 400 (→ §6 에서 재현). hosted tools(web search 등)는 Responses 전용. `temperature` 는 Provider 공통 파라미터로 쓸 수 없다(Claude 현행 모델 거부). `service_tier`(auto|default|flex|fast|ultrafast) 에 따라 단가가 달라지므로 응답의 `service_tier` 를 함께 기록한다.
- **출처**: https://developers.openai.com/api/reference/resources/responses/methods/create · https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create · https://developers.openai.com/api/docs/guides/migrate-to-responses

In [ ]:
# 💸 유료 2회 (Responses 1회 + Chat Completions 1회, 각 max_output_tokens=48)
PROMPT = "대한민국의 수도를 한 문장으로 알려줘."
SYSTEM = "You are concise. 한국어로 답한다."

if BLOCKED:
    blocked_banner("② 모델 호출"); r_resp = r_chat = None
else:
    r_resp = adapter.generate(PROMPT, system=SYSTEM, max_output_tokens=48, api="responses")
    r_chat = adapter.generate(PROMPT, system=SYSTEM, max_output_tokens=48, api="chat_completions")
    rows = []
    for r in (r_resp, r_chat):
        rows.append({"api": r.api, "status": r.status, "text": r.text, "finish_reason": r.finish_reason, "latency_ms": r.latency_ms,
                     "service_tier": r.service_tier, "request_id": r.request_id,
                     "input_tokens": r.usage.input_tokens if r.usage else None, "output_tokens": r.usage.output_tokens if r.usage else None,
                     "error": r.error.category if r.error else None})
    display(pd.DataFrame(rows).set_index("api").T)

In [ ]:
# 원본 객체/필드 확인: 두 API 의 응답 구조 차이
if BLOCKED:
    blocked_banner("② raw 구조 비교")
else:
    def _keys(d): return sorted(d.keys()) if isinstance(d, dict) else d
    print("Responses raw keys       :", _keys(r_resp.raw))
    print("Chat Completions raw keys:", _keys(r_chat.raw))
    diff = pd.DataFrame([
        {"관점": "엔드포인트", "Responses": "POST /v1/responses", "Chat Completions": "POST /v1/chat/completions"},
        {"관점": "입력", "Responses": "input=str|items, instructions=", "Chat Completions": "messages=[{role,content}]"},
        {"관점": "출력 길이", "Responses": "max_output_tokens (>=16)", "Chat Completions": "max_completion_tokens"},
        {"관점": "본문 텍스트 위치", "Responses": "output[].content[].text (output_text 편의 속성)", "Chat Completions": "choices[0].message.content"},
        {"관점": "종료 사유", "Responses": f"status={r_resp.raw.get('status')} / incomplete_details={r_resp.raw.get('incomplete_details')}", "Chat Completions": f"choices[0].finish_reason={r_chat.finish_reason}"},
        {"관점": "usage 필드명", "Responses": "input_tokens / output_tokens / total_tokens", "Chat Completions": "prompt_tokens / completion_tokens / total_tokens"},
        {"관점": "Structured Outputs", "Responses": "text.format", "Chat Completions": "response_format"},
        {"관점": "응답 id 접두", "Responses": str(r_resp.raw.get("id"))[:5], "Chat Completions": str(r_chat.raw.get("id"))[:8]},
        {"관점": "실측 model", "Responses": r_resp.raw.get("model"), "Chat Completions": r_chat.raw.get("model")},
        {"관점": "service_tier", "Responses": r_resp.service_tier, "Chat Completions": r_chat.service_tier},
    ])
    display(diff.set_index("관점"))
    print("\nResponses output[] 항목 type:", [o.get("type") for o in r_resp.raw.get("output", [])])

**관찰 메모: (실행 후 채우기)**
- 두 API 의 응답 텍스트가 같은가 / latency 차이:
- Responses `status`, `incomplete_details`:
- 응답 `model` 이 요청 모델명과 같은가(날짜 스냅샷 id 여부):

In [ ]:
if BLOCKED:
    save_blocked("02_call")
else:
    ok = r_resp.status == "ok"
    payload = {"prompt": PROMPT, "system": SYSTEM, "responses": to_dict(r_resp), "chat_completions": to_dict(r_chat),
               "structure_diff": diff.to_dict(orient="records")}
    print("evidence ->", save_evidence(PROVIDER, "02_call", payload, "live" if ok else "blocked",
                                       note="" if ok else (r_resp.error.category if r_resp.error else "error"), model=MODEL))

## 3. Token Usage

- **공식 방법**: 매 응답의 `usage` 객체.
  - Responses: `input_tokens`, `output_tokens`, `total_tokens`, `input_tokens_details.{cached_tokens, cache_write_tokens}`, `output_tokens_details.reasoning_tokens` (SDK 타입에서 전부 필수 int).
  - Chat Completions: `prompt_tokens`, `completion_tokens`, `total_tokens`, `prompt_tokens_details.{cached_tokens, cache_write_tokens, audio_tokens}`, `completion_tokens_details.{reasoning_tokens, ...}` (details 하위는 Optional → None 방어).
  - 사전 카운트: `POST /v1/responses/input_tokens` (SDK `client.responses.input_tokens.count(model=, input=)`) → `{object: "response.input_tokens", input_tokens}`.
- **주의점**: `reasoning_tokens` 는 `output_tokens` 에 **이미 포함**되어 output 단가로 과금된다(더하지 말 것). `total_tokens` 가 응답에 있다(Anthropic 은 없음). 두 API 의 필드명이 달라 정규화 레이어가 필요하다 → 어댑터의 `UsageNormalized`(과금 관점: input=캐시 포함 총입력, output=reasoning 포함 총출력). Chat 스트리밍은 `stream_options={"include_usage": True}` 를 줘야 마지막 chunk 에 usage 가 온다. `input_tokens` 카운트 엔드포인트의 과금·rate limit 여부는 문서에 명시 없음.
- **출처**: https://developers.openai.com/api/docs/guides/reasoning · https://developers.openai.com/api/docs/guides/token-counting · https://developers.openai.com/api/reference/python/resources/responses/methods/create

In [ ]:
# §2 결과 재사용 (추가 유료 호출 없음). count_tokens 는 POST /v1/responses/input_tokens (토큰 생성 없음)
if BLOCKED:
    blocked_banner("③ Token Usage"); pre = None
else:
    pre = adapter.count_tokens(PROMPT, model=MODEL)       # instructions 제외, 사용자 입력만 카운트
    print("사전 카운트(input_tokens):", to_dict(pre) if isinstance(pre, NotSupported) else pre)
    rows = []
    for r in (r_resp, r_chat):
        if r.usage:
            u = to_dict(r.usage); u.pop("raw", None); u["api"] = r.api; rows.append(u)
    if rows:
        display(pd.DataFrame(rows).set_index("api").T)
    else:
        print("§2 호출이 실패해 usage 가 없다:", to_dict(r_resp.error), to_dict(r_chat.error))
    print("검산 total == input + output:", all(r.usage.total_tokens == r.usage.input_tokens + r.usage.output_tokens for r in (r_resp, r_chat) if r.usage))

In [ ]:
# 원본 usage 객체 (필드명이 다름을 눈으로 확인)
if BLOCKED:
    blocked_banner("③ raw usage")
else:
    print("Responses usage.raw        :", json.dumps(r_resp.usage.raw if r_resp.usage else None, ensure_ascii=False))
    print("Chat Completions usage.raw :", json.dumps(r_chat.usage.raw if r_chat.usage else None, ensure_ascii=False))

**관찰 메모: (실행 후 채우기)**
- 사전 카운트 vs 실제 `input_tokens` 차이(instructions 포함 여부):
- `reasoning_tokens` 가 0 인가(gpt-5.4-mini 기본 effort=none):
- Chat `prompt_tokens_details` 가 None 으로 오는가:

In [ ]:
if BLOCKED:
    save_blocked("03_usage")
else:
    ok = bool(r_resp.usage)
    payload = {"count_tokens_pre_call": to_dict(pre),
               "responses": {"normalized": to_dict(r_resp.usage), "raw": r_resp.usage.raw if r_resp.usage else None},
               "chat_completions": {"normalized": to_dict(r_chat.usage), "raw": r_chat.usage.raw if r_chat.usage else None}}
    print("evidence ->", save_evidence(PROVIDER, "03_usage", payload, "live" if ok else "blocked", note="" if ok else "call failed", model=MODEL))

## 4. Usage 조회

- **공식 방법**: Usage API `GET /v1/organization/usage/completions` (embeddings/images/moderations/audio_*/vector_stores/code_interpreter_sessions/web_search_calls/file_search_calls/decisions 별 엔드포인트 별도). 파라미터 `start_time`(필수, Unix초), `end_time`, `bucket_width`(1m|1h|1d), `limit`, `group_by`(project_id|user_id|api_key_id|model), 필터 `project_ids/models/...`, 커서 `page`. 응답 `data[].{start_time, end_time, results[].{input_tokens, output_tokens, input_cached_tokens, input_cache_write_tokens, num_model_requests, model, project_id, ...}}`, `has_more`, `next_page`.
- **인증**: **Admin API 키 필수**(조직 Owner 가 Organization settings → Admin keys 에서 생성, `OPENAI_ADMIN_KEY`). 공식 근거: Administration overview "To access these endpoints, create an Admin API key" / "Admin API keys cannot be used for non-administration endpoints". SDK 는 Admin 엔드포인트에 `admin_api_key` 만 쓰므로(`_client.py`, `security={"admin_api_key_auth": True}`) 표준 키 클라이언트로는 서버에 가기도 전에 `TypeError("Could not resolve authentication method")` 가 난다.
- **실측(2026-10-19)**: 표준(프로젝트) 키를 Bearer 로 넣어 같은 엔드포인트를 호출하면 **403 `PermissionDeniedError` — "You have insufficient permissions for this operation. Missing scopes: api.usage.read."** (바디가 `{"error": "<문장>"}` 형태로 type/code 없음). 커뮤니티의 "404" 보고와 다르다 → 아래 `probe_usage_api_with_standard_key()` 로 각자 재확인.
- **대안**: platform.openai.com/usage 대시보드(단, Usage 조회 권한은 Owner 프리셋에만 포함될 수 있음) 또는 호출마다 응답 usage 를 자체 적재하는 `apicompare.usagelog.aggregate()` — Multi-LLM Simulator 는 이 방식을 기본으로 한다.
- **출처**: https://developers.openai.com/api/reference/python/resources/admin/subresources/organization/subresources/usage/methods/completions · https://developers.openai.com/cookbook/examples/completions_usage_api · https://developers.openai.com/api/docs/guides/administration

In [ ]:
from datetime import datetime, timedelta, timezone
now = datetime.now(timezone.utc)
START, END = now - timedelta(days=7), now

if BLOCKED:
    blocked_banner("④ Usage 조회"); rep = probe = None
else:
    rep = adapter.usage_report(START, END, bucket="1d", group_by=("model",))
    if isinstance(rep, NotSupported):
        print(f"[BLOCKED: no admin key] {rep.reason}\n  대안: {rep.alternative}")
        print("\n표준 키로 같은 엔드포인트를 1회 호출해 실제 응답을 실측한다 (토큰 과금 없음):")
        probe = adapter.probe_usage_api_with_standard_key()
        show(probe)
    else:
        probe = None
        print(f"Admin API 버킷 {len(rep.buckets)}개 — {rep.note}")
        display(pd.DataFrame([{**b.group, "start": b.start, "end": b.end, "input": b.input_tokens, "output": b.output_tokens, "requests": b.requests} for b in rep.buckets]))

local = usagelog.aggregate(START, END)      # 자체 적재분 (data/usage_log.jsonl, gitignored)
print(f"\nusagelog 자체 집계: {local.note}")
if local.buckets:
    display(pd.DataFrame([{**b.group, "input": b.input_tokens, "output": b.output_tokens, "requests": b.requests, "cost_usd": b.cost_usd} for b in local.buckets]))

In [ ]:
# 원본 확인: NotSupported / probe 의 상태코드·code, 또는 Admin 버킷 raw 첫 항목
if BLOCKED:
    blocked_banner("④ raw")
elif isinstance(rep, NotSupported):
    print("NotSupported:", to_dict(rep))
    print("probe:", {k: getattr(probe, k) for k in ("category", "status_code", "provider_type", "provider_code", "exception_class", "request_id")})
    print("probe.message:", probe.message)
else:
    show(rep.buckets[0].raw if rep.buckets else rep)

**관찰 메모: (실행 후 채우기)**
- Admin 키 보유 여부 / `usage_report()` 결과(LIVE 또는 BLOCKED):
- 표준 키 프로브의 실측 상태코드·메시지(403 Missing scopes: api.usage.read 가 재현되는가):
- 킥오프 질문: 기업 발급 키가 Admin 키를 포함하는가, 아니면 자체 적재(usagelog)로 가는가

In [ ]:
if BLOCKED:
    save_blocked("04_usage_api")
else:
    if isinstance(rep, NotSupported):
        payload = {"not_supported": to_dict(rep), "probe_with_standard_key": to_dict(probe), "usagelog_local": to_dict(local)}
        print("evidence ->", save_evidence(PROVIDER, "04_usage_api", payload, "blocked", note=f"no admin key; standard key -> HTTP {probe.status_code} {probe.category}", model=MODEL))
    else:
        payload = {"usage_report": to_dict(rep), "usagelog_local": to_dict(local)}
        print("evidence ->", save_evidence(PROVIDER, "04_usage_api", payload, "live", model=MODEL))

## 5. 비용 확인

- **공식 방법 ① 로컬 계산**: 단가는 **어떤 API 응답에도 없다**(`/v1/models` 에 price 없음) → 공식 pricing 페이지를 날짜·출처와 함께 로컬 테이블(`apicompare.pricing.PRICE_TABLE`)로 관리.
  `cost = (input − cached − cache_write)×in + cached×cached_in + cache_write×in×1.25(GPT-5.6+) + output(reasoning 포함)×out`, 단위 $/1M.
  gpt-5.4-mini: $0.75 / $0.075(cached) / $4.50 (2026-10-18). Flex·Batch 는 별도 표(대략 50%), Fast(구 Priority) 약 2배 → 응답의 `service_tier` 를 같이 기록.
- **공식 방법 ② Costs API**: `GET /v1/organization/costs` — `start_time`(필수), `bucket_width="1d"` 만, `group_by`(project_id|user_id|line_item|api_key_id|api_source), 응답 `results[].{amount.{value(달러 float), currency("usd")}, line_item, project_id, ...}`. **Admin 키 필수**, 모델별 group_by 없음(`line_item` 으로 간접 구분). 데이터 반영 지연은 문서에 명시 없음.
- **주의점**: 단가가 자주 바뀌고 신규 모델명이 많아 가격표에 `as_of`·출처를 남긴다. 리전(data residency) 엔드포인트는 10% 가산. Spend Limit API(`/organization/spend_limit`) 의 `threshold_amount` 는 센트 단위(int) 인 반면 Costs 의 `amount.value` 는 달러(float).
- **출처**: https://developers.openai.com/api/docs/pricing · https://developers.openai.com/api/reference/typescript/resources/admin/subresources/organization/subresources/usage/methods/costs · https://developers.openai.com/api/docs/guides/prompt-caching

In [ ]:
if BLOCKED:
    blocked_banner("⑤ 비용 확인"); cost_resp = cost_chat = crep = None
else:
    cost_resp = adapter.cost(MODEL, r_resp.usage, service_tier=r_resp.service_tier) if r_resp.usage else None
    cost_chat = adapter.cost(MODEL, r_chat.usage, service_tier=r_chat.service_tier) if r_chat.usage else None
    for label, c in (("responses", cost_resp), ("chat_completions", cost_chat)):
        print(f"{label:17s} 예상 비용:", (f"${c.usd:.8f}  {c.breakdown}  (단가 {c.price_as_of})" if c and not isinstance(c, NotSupported) else to_dict(c)))
    # 자체 적재 (④ 의 usagelog.aggregate 가 읽는 파일, gitignored)
    for r, c in ((r_resp, cost_resp), (r_chat, cost_chat)):
        if r.status == "ok":
            usagelog.append(r, c if c and not isinstance(c, NotSupported) else None, run_id="nb10")
    crep = adapter.cost_report(START, END)
    if isinstance(crep, NotSupported):
        print(f"\n[BLOCKED: no admin key] cost_report: {crep.reason}\n  대안: {crep.alternative}")
    else:
        display(pd.DataFrame([{**b.group, "start": b.start, "cost_usd": b.cost_usd} for b in crep.buckets]))

In [ ]:
# 원본 확인: 사용한 단가 행 + CostEstimate 전체
price = pricing.get_price(PROVIDER, MODEL)
print("PRICE_TABLE row:", to_dict(price))
if not BLOCKED:
    show(cost_resp)

**관찰 메모: (실행 후 채우기)**
- 1회 호출 예상 비용(USD) / 두 API 간 차이:
- Costs API 사용 가능 여부(Admin 키):
- 기업 '모델별 비용 계산 기준정보' 와 단가표가 일치하는지(킥오프 후 확인):

In [ ]:
if BLOCKED:
    save_blocked("05_costs")
else:
    ok = cost_resp is not None and not isinstance(cost_resp, NotSupported)
    payload = {"local_estimate": {"responses": to_dict(cost_resp), "chat_completions": to_dict(cost_chat)},
               "price_row": to_dict(price), "cost_report": to_dict(crep)}
    print("evidence ->", save_evidence(PROVIDER, "05_costs", payload, "live" if ok else "blocked", note="" if ok else "no price", model=MODEL))

## 6. 상태 및 오류 확인

- **헬스체크**: `GET /v1/models`(토큰 소비 없음)로 키·네트워크 확인 → `adapter.health()`. 생성 경로까지 보려면 `max_output_tokens` 최소의 `responses.create`.
- **서비스 상태**: https://status.openai.com — Statuspage API `GET /api/v2/status.json`(전체 indicator), `/api/v2/components.json`(Responses, Chat Completions 등 컴포넌트별), 인증 불필요 → `adapter.status_page()`.
- **오류 taxonomy** (HTTP → 바디 `{error:{message, type, code, param}}`): 400 `invalid_request_error`(파라미터 오류, Chat Completions 의 tools+reasoning_effort 포함) / 401 Incorrect API key(`AuthenticationError`) / 403 지역·권한(`PermissionDeniedError`) / 404 `NotFoundError` / 429 `insufficient_quota`+`credit_balance_exhausted`(크레딧 소진) vs `rate_limit_error`(RPM/TPM, Retry-After) vs `*_spend_limit_exceeded` — **같은 429 라서 type/code 로 구분** / 500 / 503 `server_is_overloaded`.
- **Rate-limit 헤더**: `x-ratelimit-limit-requests`, `x-ratelimit-limit-tokens`, `x-ratelimit-remaining-requests`, `x-ratelimit-remaining-tokens`, `x-ratelimit-reset-requests`(예 "1s"), `x-ratelimit-reset-tokens`(예 "6m0s"), 프로젝트 범위 3종, `Retry-After`. 디버깅용 `x-request-id`, `openai-processing-ms`. `with_raw_response` 로 읽는다(어댑터는 화이트리스트만 `response_headers` 에 저장).
- **SDK 예외**: `APIError` ← `APIConnectionError` ← `APITimeoutError`; `APIStatusError(.status_code, .response, .body, .request_id)` ← BadRequest(400)/Authentication(401)/PermissionDenied(403)/NotFound(404)/Conflict(409)/UnprocessableEntity(422)/RateLimit(429)/InternalServer(≥500). 설정 `OpenAI(timeout=, max_retries=)` — 기본 600초·2회, 과금/쿼터 429 는 재시도 금지, `x-should-retry` 헤더 우선, `Retry-After`>120초면 재시도 안 함.
- **출처**: https://developers.openai.com/api/docs/guides/error-codes · https://developers.openai.com/api/docs/guides/rate-limits · https://status.openai.com/api/v2/status.json · https://github.com/openai/openai-python#handling-errors

In [ ]:
# 헬스체크 + 상태 페이지 (무료)
if BLOCKED:
    blocked_banner("⑥ health")
h = adapter.health()                      # NO_KEY 면 ok=False, error=AUTH
print("health:", {k: v for k, v in to_dict(h).items() if k != "ratelimit"})
sp = adapter.status_page()                # 인증 불필요 → NO_KEY 에서도 동작
print("\nstatus.openai.com:", sp.get("indicator"), "-", sp.get("description"), sp.get("error", ""))
api_components = [c for c in sp.get("components", []) if any(w in (c.get("name") or "") for w in ("Responses", "Chat", "API", "Realtime", "Batch", "Embeddings", "Fine"))]
display(pd.DataFrame(api_components or sp.get("components", [])[:15]))

In [ ]:
# 의도적 오류 4종 (모두 서버가 처리 전 거부 → 토큰 과금 없음) + x-ratelimit-* 헤더 표
if BLOCKED:
    blocked_banner("⑥ 의도적 오류 재현"); errors = {}
else:
    errors = {}
    bad_key = get_adapter("openai", api_key="INVALID-KEY-FOR-TEST", admin_key="", timeout_s=adapter.timeout_s)
    errors["401 bad_key"] = bad_key.generate("ping", model=MODEL, max_output_tokens=16).error
    errors["404 bad_model"] = adapter.generate("ping", model="model-does-not-exist-123", max_output_tokens=16).error   # max_output_tokens<16 이면 400 integer_below_min_value 가 먼저 난다
    quick = get_adapter("openai", timeout_s=0.001)
    errors["timeout 0.001s"] = quick.generate("ping", model=MODEL, max_output_tokens=16).error
    # GPT-5.4+ Chat Completions: tools + reasoning_effort≠none → 400 invalid_request_error (매트릭스 기록; 메시지는 실행 후 메모에 채우기)
    noop_tool = [{"type": "function", "function": {"name": "noop", "description": "does nothing", "parameters": {"type": "object", "properties": {}}}}]
    errors["400 chat tools+reasoning"] = adapter.generate("ping", model=MODEL, api="chat_completions", max_output_tokens=16, tools=noop_tool, reasoning_effort="low").error
    display(pd.DataFrame([{"case": k, **{f: getattr(e, f) for f in ("category", "status_code", "provider_type", "provider_code", "exception_class", "retry_after")}, "message": (e.message or "")[:110]}
                          for k, e in errors.items() if e is not None]).set_index("case"))

    # x-ratelimit-* 헤더: 생성 호출(§2)의 응답 헤더에서 읽는다 (/v1/models 응답에는 x-ratelimit-* 가 없다는 것을 health.ratelimit 로 확인)
    hdr_rows = []
    for label, hd in (("responses(§2)", r_resp.response_headers), ("chat_completions(§2)", r_chat.response_headers), ("GET /v1/models(health)", h.ratelimit)):
        for k, v in sorted(hd.items()):
            if k.startswith("x-ratelimit-") or k in ("retry-after", "openai-processing-ms", "x-request-id"):
                hdr_rows.append({"source": label, "header": k, "value": v})
    if hdr_rows:
        display(pd.DataFrame(hdr_rows).pivot_table(index="header", columns="source", values="value", aggfunc="first"))
    else:
        print("화이트리스트 헤더가 비어 있음 (§2 호출 실패?)")

**관찰 메모: (실행 후 채우기)**
- health latency / status.openai.com indicator:
- 401 의 `code`(invalid_api_key?) / 404 의 `code`(model_not_found?) — 공식 error-codes 페이지에 없는 문자열 실측:
- timeout 0.001s 의 분류(2026-10-19 실측: macOS 에서 `ConnectError: No route to host` → `APIConnectionError` → NETWORK. 읽기 타임아웃을 보려면 `timeout_s=httpx2.Timeout(connect=10, read=0.05, write=10, pool=10)` 처럼 connect 는 넉넉히, read 만 짧게):
- Chat Completions tools+reasoning_effort 400 의 실제 메시지:
- `x-ratelimit-remaining-*` 값과 `reset` 단위:

In [ ]:
payload = {"health": to_dict(h), "status_page": {k: v for k, v in sp.items() if k != "components"},
           "status_components_api": api_components,
           "intentional_errors": {k: to_dict(e) for k, e in errors.items()},
           "ratelimit_headers": hdr_rows if not BLOCKED else []}
if BLOCKED:
    print("evidence ->", save_evidence(PROVIDER, "06_errors", payload, "blocked", note="no key", model=MODEL))
else:
    print("evidence ->", save_evidence(PROVIDER, "06_errors", payload, "live" if h.ok else "blocked", note="" if h.ok else (h.error.category if h.error else "health failed"), model=MODEL))

## 7. 호출 예제

- **공식 방법**: `pip install openai` → `export OPENAI_API_KEY=...` → `OpenAI().responses.create(model=, input=)` → `response.output_text`, `response.usage`. 어댑터 없이 SDK 를 직접 호출하는 최소 예제와, 같은 요청의 curl.
- **주의점**: quickstart 의 모델명은 문서 시점마다 바뀐다(현재 gpt-6-astra). 저비용 실습 모델은 gpt-5.4-mini(종료 미공지) 또는 gpt-6-luna($0.10/$0.01/$0.50). gpt-5-nano/gpt-5-mini 는 2026-12-11 종료 예정이라 PoC 기간상 비권장. 키는 코드에 하드코딩하지 않는다.
- **출처**: https://developers.openai.com/api/docs/quickstart · https://github.com/openai/openai-python

In [ ]:
# 💸 유료 1회 (max_output_tokens=32). 어댑터 없이 SDK 직접 호출 — 10줄 이하
if BLOCKED:
    blocked_banner("⑦ 호출 예제"); resp = None
else:
    from openai import OpenAI
    client = OpenAI(timeout=30.0, max_retries=0)          # OPENAI_API_KEY 환경변수 사용(load_env 로 로드됨)
    resp = client.responses.create(model=MODEL, input=PROMPT, instructions=SYSTEM, max_output_tokens=32, store=False)
    print(resp.output_text)
    print("usage:", resp.usage.input_tokens, resp.usage.output_tokens, resp.usage.total_tokens, "| status:", resp.status, "| id:", resp.id[:12] + "...")

In [ ]:
# 동일 요청 curl (키는 $OPENAI_API_KEY 환경변수 표기 — 값 출력 금지)
CURL = f'''curl https://api.openai.com/v1/responses \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer $OPENAI_API_KEY" \\
  -d '{json.dumps({"model": MODEL, "input": PROMPT, "instructions": SYSTEM, "max_output_tokens": 32, "store": False}, ensure_ascii=False)}\''''
print(CURL)
CURL_CHAT = f'''curl https://api.openai.com/v1/chat/completions \\
  -H "Content-Type: application/json" \\
  -H "Authorization: Bearer $OPENAI_API_KEY" \\
  -d '{json.dumps({"model": MODEL, "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": PROMPT}], "max_completion_tokens": 32}, ensure_ascii=False)}\''''
print("\n# Chat Completions 동등 예제\n" + CURL_CHAT)

**관찰 메모: (실행 후 채우기)**
- SDK 직접 호출 결과가 §2 어댑터 결과와 같은 구조인가:
- curl 로도 실행해 봤는가(터미널에서 `export OPENAI_API_KEY=...` 후):

In [ ]:
PY_EXAMPLE = '''from openai import OpenAI
client = OpenAI(timeout=30.0, max_retries=0)  # reads OPENAI_API_KEY
resp = client.responses.create(model="gpt-5.4-mini", input="대한민국의 수도를 한 문장으로 알려줘.",
                               instructions="You are concise.", max_output_tokens=32, store=False)
print(resp.output_text, resp.usage.input_tokens, resp.usage.output_tokens, resp.usage.total_tokens)'''
if BLOCKED:
    save_blocked("07_example")
else:
    payload = {"python": PY_EXAMPLE, "curl_responses": CURL, "curl_chat_completions": CURL_CHAT,
               "result": {"id": resp.id, "model": resp.model, "status": resp.status, "output_text": resp.output_text,
                          "usage": resp.usage.model_dump() if resp.usage else None}}
    print("evidence ->", save_evidence(PROVIDER, "07_example", payload, "live", model=MODEL))

## 8. 남긴 evidence

`evidence/openai/0N_*.json` 7개가 매트릭스(`matrix/<이름>.yaml`) 의 OpenAI 7셀의 근거다. 배지: `[LIVE]` 실측 / `[OBSERVED]` / `[DOCS]` / `[BLOCKED: 사유]`.
Admin 키가 없으면 ④ 는 `[BLOCKED: no admin key; standard key -> HTTP 403 PERMISSION]` 이 정상이다. 키 자체가 없으면 7개 모두 `[BLOCKED: no key]`.

In [ ]:
df = pd.DataFrame(summary(PROVIDER))
display(df)
print("\n매트릭스 YAML evidence 값 제안:", {row["item"]: ("live" if row["badge"] == "[LIVE]" else "blocked") for row in summary(PROVIDER)})